# DataPact - Multi-Model System: Part 1, Model Size Selection and Benchmarking

**INFO 490, Team 15. Part 1, Steps 1.1-1.5: Hriday Agarwal.**
(Step 1.6 analysis: Ashok. Part 2 scaling simulations: Connor. Part 3 routing: Tejas.)

DataPact checks recurring data files against versioned *data contracts*. This notebook benchmarks
the **same 15 models from A6** on **25 realistic DataPact prompts** (5 prompt types x 5 prompts),
measuring **speed** (tokens per second, latency, time to first token) and **output quality**
(a 1-10 self-evaluation score from a fixed rubric).

**How to run:** in Google Colab choose *Runtime > Change runtime type > T4 GPU*, then
*Runtime > Run all*. It takes roughly 45-60 minutes. If the session crashes, *Run all* again:
finished models are skipped. Setting the environment variable `SMOKE_TEST=1` runs 2 models on
5 prompts for a quick check.

## Install libraries

* `transformers`: downloads a model by name and runs it.
* `accelerate`: places the model on the GPU.

PyTorch is already installed on Colab.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
packages = ["transformers==5.20.1", "accelerate==1.15.0", "pandas", "matplotlib"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Running in Colab:", IN_COLAB)

## Step 1.1: Model size categories

These are the 15 models from A6 (`llm-test/ai_prototype_light.ipynb`), all openly downloadable,
non-gated instruct/chat models. Parameter counts are the real totals from each model's
`safetensors` metadata. In A6 the free Colab session crashed on 7-8B models, so the team's A6
model set stays under 2B parameters, and the categories split that range into five groups of three:

| Model Category | Parameter Range | Models |
|---|---|---|
| Ultra-light | 0.1B - 0.4B | SmolLM2-135M, SmolLM-360M, SmolLM2-360M |
| Light | 0.4B - 0.6B | Qwen2-0.5B, Qwen2.5-0.5B, H2O-Danube3-500M |
| Small | 1.0B - 1.5B | TinyLlama-1.1B, Granite-3.1-1B-A400M, OLMo-2-1B |
| Medium | 1.5B - 1.7B | Qwen2.5-1.5B, StableLM-2-Zephyr-1.6B, Falcon3-1B |
| Large | 1.7B - 2.0B | SmolLM2-1.7B, H2O-Danube2-1.8B, Qwen1.5-1.8B |

"Large" means the largest of these models, not a 7B+ model.

In [ ]:
import os

import pandas as pd

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"

MODELS = [
    # (category, Hugging Face id, parameters in billions)
    ("Ultra-light", "HuggingFaceTB/SmolLM2-135M-Instruct", 0.13),
    ("Ultra-light", "HuggingFaceTB/SmolLM-360M-Instruct", 0.36),
    ("Ultra-light", "HuggingFaceTB/SmolLM2-360M-Instruct", 0.36),
    ("Light", "Qwen/Qwen2-0.5B-Instruct", 0.49),
    ("Light", "Qwen/Qwen2.5-0.5B-Instruct", 0.49),
    ("Light", "h2oai/h2o-danube3-500m-chat", 0.51),
    ("Small", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", 1.10),
    ("Small", "ibm-granite/granite-3.1-1b-a400m-instruct", 1.33),
    ("Small", "allenai/OLMo-2-0425-1B-Instruct", 1.48),
    ("Medium", "Qwen/Qwen2.5-1.5B-Instruct", 1.54),
    ("Medium", "stabilityai/stablelm-2-zephyr-1_6b", 1.64),
    ("Medium", "tiiuae/Falcon3-1B-Instruct", 1.67),
    ("Large", "HuggingFaceTB/SmolLM2-1.7B-Instruct", 1.71),
    ("Large", "h2oai/h2o-danube2-1.8b-chat", 1.83),
    ("Large", "Qwen/Qwen1.5-1.8B-Chat", 1.84),
]
CATEGORY_ORDER = ["Ultra-light", "Light", "Small", "Medium", "Large"]
if SMOKE_TEST:
    MODELS = [m for m in MODELS if m[1] in (
        "HuggingFaceTB/SmolLM2-135M-Instruct", "Qwen/Qwen2.5-0.5B-Instruct")]

import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

categories = pd.DataFrame(MODELS, columns=["Model Category", "Model", "Parameters (B)"])
print("Device:", DEVICE, "| smoke test:", SMOKE_TEST, "| models:", len(MODELS))
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
categories

## Step 1.2: Prompt types

The five prompt types are the five things DataPact would ask a model to do:

| # | Prompt Type | What DataPact needs | Expected output |
|---|---|---|---|
| 1 | **Classification** | Decide whether a failed check means bad data or an out-of-date contract | JSON: `likely_cause` + `reason` |
| 2 | **Structured extraction** | Turn a data steward's plain-English rule request into contract fields | JSON: `dataset`, `column`, `rule_type`, `severity` |
| 3 | **Summarization** | Explain a whole validation run to a manager | At most 2 sentences, plain text |
| 4 | **Prediction** | Forecast the next run from the recent pass-rate history | JSON: `trend` + `next_run` |
| 5 | **Text generation** | Draft the email asking a data provider to fix a problem | 3-5 sentence email |

Three of the five return JSON because DataPact's code has to read those answers automatically.

## Step 1.3: The 25 evaluation prompts

Each prompt is a realistic input written in DataPact's own vocabulary (datasets, columns, rule
types such as `RANGE`, `ALLOWED_VALUES`, `UNIQUE`, `NOT_NULL`, `REGEX`, `TYPE_MATCH`).
**Every prompt has a known correct answer**, so outputs can be scored the same way for every model.

For the prediction prompts a run counts as passing when at least 98.0% of rows pass. The correct
trend and next result follow from the numbers by construction: steady rises or falls of about
0.3-0.7 points per run, or a flat series that varies by 0.2 points or less.

In [ ]:
INSTRUCTIONS = {
    "Classification": (
        "You review failed data-quality checks for DataPact. A failure means either the incoming "
        "data is wrong (DATA_ISSUE) or the data is legitimate and the rule is out of date "
        "(CONTRACT_UPDATED).\n"
        "Reply with ONLY this JSON and no other text:\n"
        '{"likely_cause": "DATA_ISSUE" or "CONTRACT_UPDATED", "reason": "<one short sentence>"}'),
    "Structured extraction": (
        "Extract the data-contract rule described in this message from a data steward.\n"
        "rule_type must be one of: RANGE, ALLOWED_VALUES, REGEX, UNIQUE, NOT_NULL, TYPE_MATCH.\n"
        "severity must be ERROR (blocks the upload) or WARNING (only flags it).\n"
        "Reply with ONLY this JSON and no other text:\n"
        '{"dataset": "...", "column": "...", "rule_type": "...", "severity": "..."}'),
    "Summarization": (
        "Summarize this validation run for a manager in at most 2 sentences (60 words maximum). "
        "Mention the overall result, how many rules failed, and the failure that affected the "
        "most rows. Plain text only, no lists or JSON."),
    "Prediction": (
        "These are the row pass rates of the last six validation runs, oldest first. A run passes "
        "when at least 98.0% of rows pass. Predict the trend and whether the next run will pass if "
        "the trend continues.\n"
        "Reply with ONLY this JSON and no other text:\n"
        '{"trend": "IMPROVING" or "DECLINING" or "STABLE", "next_run": "PASS" or "FAIL"}'),
    "Text generation": (
        "Write a short email (3 to 5 sentences) to the data provider asking them to fix the problem "
        "below. Start with \"Hello\", mention the dataset, the column and the number of failed "
        "rows, say what they should do, and sign off as \"DataPact\"."),
}

ACTION = r"\b(correct|fix|update|resend|re-send|re-export|reexport|re-upload|review|investigate|check|remove|fill|replace)\w*"

PROMPTS = [
    # ---------------- Classification ----------------
    dict(id="C1", type="Classification", truth={"likely_cause": "DATA_ISSUE"}, input=(
        "Dataset: Weekly Sales Extract\nColumn: unit_price\nRule: RANGE {\"min\": 0.01, \"max\": 5000}\n"
        "Rows failed: 23 of 13,120\nSample values: [-15.00, 0.00, -4.99]")),
    dict(id="C2", type="Classification", truth={"likely_cause": "CONTRACT_UPDATED"}, input=(
        "Dataset: Monthly Enrollment Export\nColumn: residency\n"
        "Rule: ALLOWED_VALUES [\"IN_STATE\", \"OUT_OF_STATE\", \"INTERNATIONAL\"]\n"
        "Rows failed: 380 of 4,812\nSample values: [\"ONLINE_ONLY\", \"ONLINE_ONLY\", \"ONLINE_ONLY\"]")),
    dict(id="C3", type="Classification", truth={"likely_cause": "DATA_ISSUE"}, input=(
        "Dataset: Sensor Readings - Lab 3\nColumn: timestamp\nRule: NOT_NULL\n"
        "Rows failed: 31 of 8,640\nSample values: [null, null, null]")),
    dict(id="C4", type="Classification", truth={"likely_cause": "CONTRACT_UPDATED"}, input=(
        "Dataset: Facilities Work Orders\nColumn: building_code\n"
        "Rule: ALLOWED_VALUES [\"SIEBEL\", \"ARC\", \"UNION\"]\n"
        "Rows failed: 260 of 2,210\nSample values: [\"CIF\", \"CIF\", \"CIF\"]")),
    dict(id="C5", type="Classification", truth={"likely_cause": "DATA_ISSUE"}, input=(
        "Dataset: Payroll Deductions Export\nColumn: amount\nRule: TYPE_MATCH {\"type\": \"number\"}\n"
        "Rows failed: 3 of 3,402\nSample values: [\"12..50\", \"N/A\", \"$\"]")),
    # ---------------- Structured extraction ----------------
    dict(id="E1", type="Structured extraction", input=(
        "Hi, for the Library Circulation Log, every checkout_id must appear only once. Treat any "
        "duplicates as an error that blocks the upload. Thanks, Maria"),
        truth={"dataset": "Library Circulation Log", "column": "checkout_id",
               "rule_type": "UNIQUE", "severity": "ERROR"}),
    dict(id="E2", type="Structured extraction", input=(
        "Quick request: in the Sensor Readings - Lab 3 file, humidity_pct should always be between "
        "0 and 100. If it isn't, just flag a warning, don't block the load."),
        truth={"dataset": "Sensor Readings - Lab 3", "column": "humidity_pct",
               "rule_type": "RANGE", "severity": "WARNING"}),
    dict(id="E3", type="Structured extraction", input=(
        "Starting next month the Monthly Enrollment Export must never have an empty advisor_email. "
        "Missing values should fail the run."),
        truth={"dataset": "Monthly Enrollment Export", "column": "advisor_email",
               "rule_type": "NOT_NULL", "severity": "ERROR"}),
    dict(id="E4", type="Structured extraction", input=(
        "For the Course Catalog Snapshot, course_code has to look like four capital letters followed "
        "by three digits, for example INFO490. Mismatches should only be flagged as warnings."),
        truth={"dataset": "Course Catalog Snapshot", "column": "course_code",
               "rule_type": "REGEX", "severity": "WARNING"}),
    dict(id="E5", type="Structured extraction", input=(
        "Payroll Deductions Export: pay_frequency may only be WEEKLY, BIWEEKLY or MONTHLY. Anything "
        "else is an error and must block the upload."),
        truth={"dataset": "Payroll Deductions Export", "column": "pay_frequency",
               "rule_type": "ALLOWED_VALUES", "severity": "ERROR"}),
    # ---------------- Summarization ----------------
    dict(id="S1", type="Summarization", input=(
        "Validation run #412 - Monthly Enrollment Export (4,812 rows)\nRules checked: 6. Passed: 5. Failed: 1.\n"
        "- FAILED credit_hours RANGE {min 0, max 24}: 47 rows (samples 31, 99, -2)\nOverall result: FAILED"),
        truth={"facts": [r"enrol", r"fail", r"\b(1|one|single)\b", r"credit[_ ]hours"]}),
    dict(id="S2", type="Summarization", input=(
        "Validation run #88 - Sensor Readings - Lab 3 (8,640 rows)\nRules checked: 8. Passed: 6. Failed: 2.\n"
        "- FAILED temperature_c RANGE {min -10, max 50}: 6 rows (samples -9999)\n"
        "- FAILED timestamp NOT_NULL: 31 rows\nOverall result: FAILED"),
        truth={"facts": [r"sensor", r"fail", r"\b(2|two)\b", r"timestamp"]}),
    dict(id="S3", type="Summarization", input=(
        "Validation run #231 - Library Circulation Log (9,800 rows)\nRules checked: 7. Passed: 7. Failed: 0.\n"
        "No failures.\nOverall result: PASSED"),
        truth={"facts": [r"library", r"pass", r"\b(7|seven|all|0|no|zero|none)\b"]}),
    dict(id="S4", type="Summarization", input=(
        "Validation run #57 - Payroll Deductions Export (3,402 rows)\nRules checked: 5. Passed: 4. Failed: 1.\n"
        "- FAILED deduction_code ALLOWED_VALUES [FED, STATE, MEDICARE, SOCSEC]: 512 rows (value HSA)\n"
        "Overall result: FAILED"),
        truth={"facts": [r"payroll", r"fail", r"\b(1|one|single)\b", r"deduction[_ ]code|\bhsa\b"]}),
    dict(id="S5", type="Summarization", input=(
        "Validation run #19 - Facilities Work Orders (2,210 rows)\nRules checked: 6. Passed: 4. Failed: 2.\n"
        "- FAILED order_id UNIQUE: 14 rows\n"
        "- FAILED priority ALLOWED_VALUES [LOW, MEDIUM, HIGH]: 140 rows (value URGENT)\nOverall result: FAILED"),
        truth={"facts": [r"facilit|work order", r"fail", r"\b(2|two)\b", r"priority|urgent"]}),
    # ---------------- Prediction ----------------
    dict(id="P1", type="Prediction", truth={"trend": "DECLINING", "next_run": "FAIL"},
         input="Dataset: Monthly Enrollment Export\nPass rates: 99.6%, 99.4%, 99.1%, 98.7%, 98.4%, 98.1%"),
    dict(id="P2", type="Prediction", truth={"trend": "IMPROVING", "next_run": "PASS"},
         input="Dataset: Sensor Readings - Lab 3\nPass rates: 95.2%, 96.0%, 96.9%, 97.5%, 98.2%, 98.9%"),
    dict(id="P3", type="Prediction", truth={"trend": "STABLE", "next_run": "PASS"},
         input="Dataset: Library Circulation Log\nPass rates: 99.5%, 99.6%, 99.5%, 99.4%, 99.5%, 99.5%"),
    dict(id="P4", type="Prediction", truth={"trend": "STABLE", "next_run": "FAIL"},
         input="Dataset: Payroll Deductions Export\nPass rates: 97.0%, 96.9%, 97.1%, 97.0%, 96.9%, 97.0%"),
    dict(id="P5", type="Prediction", truth={"trend": "DECLINING", "next_run": "FAIL"},
         input="Dataset: Facilities Work Orders\nPass rates: 99.9%, 99.5%, 99.2%, 98.9%, 98.5%, 98.2%"),
    # ---------------- Text generation ----------------
    dict(id="G1", type="Text generation", input=(
        "Dataset: Weekly Sales Extract\nColumn: unit_price\nProblem: 23 rows have a zero or negative price."),
        truth={"facts": [r"sales", r"unit[_ ]price", r"\b23\b", ACTION]}),
    dict(id="G2", type="Text generation", input=(
        "Dataset: Sensor Readings - Lab 3\nColumn: timestamp\nProblem: 31 rows are missing their timestamp."),
        truth={"facts": [r"sensor", r"timestamp", r"\b31\b", ACTION]}),
    dict(id="G3", type="Text generation", input=(
        "Dataset: Monthly Enrollment Export\nColumn: student_id\nProblem: 12 rows have student IDs that "
        "are not exactly 8 digits."),
        truth={"facts": [r"enrol", r"student[_ ]id", r"\b12\b", ACTION]}),
    dict(id="G4", type="Text generation", input=(
        "Dataset: Facilities Work Orders\nColumn: order_id\nProblem: 14 rows repeat an order_id that "
        "already appears in the file."),
        truth={"facts": [r"facilit|work order", r"order[_ ]id", r"\b14\b", ACTION]}),
    dict(id="G5", type="Text generation", input=(
        "Dataset: Course Catalog Snapshot\nColumn: credit_hours\nProblem: 18 rows list a course worth 0 "
        "credit hours, but every course must be worth at least 1."),
        truth={"facts": [r"catalog|course", r"credit[_ ]hours", r"\b18\b", ACTION]}),
]
PROMPT_TYPES = list(INSTRUCTIONS)
MAX_NEW_TOKENS = {"Classification": 120, "Structured extraction": 120, "Summarization": 140,
                  "Prediction": 80, "Text generation": 220}

if SMOKE_TEST:
    PROMPTS = [p for p in PROMPTS if p["id"].endswith("1")]

assert len({p["id"] for p in PROMPTS}) == len(PROMPTS)
prompt_table = pd.DataFrame([{"Prompt Type": p["type"], "ID": p["id"], "Input": p["input"]}
                             for p in PROMPTS])
print(len(PROMPTS), "prompts:", prompt_table["Prompt Type"].value_counts().to_dict())
pd.set_option("display.max_colwidth", 120)
prompt_table

## Step 1.4: Evaluation metrics

Every output gets a **self-evaluation score from 1 to 10**, built from four criteria:

| Metric | Points | Description |
|---|---|---|
| **Instruction adherence** | 3 | Did the model follow the instructions: allowed values only, exactly the requested keys, no extra text around JSON, the sentence and word limits? |
| **Accuracy** | 4 | Was the answer correct? Classification and prediction: the right label. Extraction: the share of the 4 fields that match. Summaries and emails: the share of the required facts mentioned (dataset, result or column, the key number, the main issue or the requested action). Values are compared ignoring case, spaces and underscores. An answer that mostly copies the input (over half of its 5-word sequences appear in the prompt) earns no accuracy points, because repeating the report is not summarizing it. |
| **Clarity** | 2 | Is it understandable? It does not loop or repeat itself, and it is not cut off mid-answer. For classification, the reason must be a real sentence (3+ words), not just a label. |
| **Structure** | 1 | Is it in the expected format? Valid JSON for the JSON types, plain prose for summaries, and an email that opens with "Hello" and signs off as DataPact. |

The score is `max(1, adherence + accuracy + clarity + structure)`, rounded to one decimal.

Limits used for adherence: summaries need 1-2 sentences and at most 60 words; emails need 3-7
sentences (the requested 3-5, plus the "Hello" and sign-off lines, which count as sentences when
they sit on their own line) and at most 130 words.

**Why a fixed rubric, applied by code?** 15 models x 25 prompts gives 375 answers. Scoring them
with the same written rules makes every score repeatable and comparable between models; scoring
by eye would drift. The rules were written and tested on hand-written good and bad answers
(next cell) before the benchmark. A 2-model trial run then exposed three loopholes (a label used
as the "reason", extra JSON keys, and a summary that copied the input), which were closed before
the full run; the trial's examples are among the self-tests. Sample outputs are printed at the end so the scores
can be checked by reading them.

In [ ]:
import json
import re

ALLOWED = {
    "Classification": {"likely_cause": {"DATA_ISSUE", "CONTRACT_UPDATED"}},
    "Structured extraction": {"rule_type": {"RANGE", "ALLOWED_VALUES", "REGEX", "UNIQUE",
                                            "NOT_NULL", "TYPE_MATCH"},
                              "severity": {"ERROR", "WARNING"}},
    "Prediction": {"trend": {"IMPROVING", "DECLINING", "STABLE"}, "next_run": {"PASS", "FAIL"}},
}
KEYS = {"Classification": {"likely_cause", "reason"},
        "Structured extraction": {"dataset", "column", "rule_type", "severity"},
        "Prediction": {"trend", "next_run"}}


def is_looping(text):
    # True when many 4-word sequences repeat: the model is stuck in a loop.
    words = re.findall(r"\w+", text.lower())
    grams = [tuple(words[i:i + 4]) for i in range(len(words) - 3)]
    return len(grams) >= 8 and 1 - len(set(grams)) / len(grams) > 0.3


def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if re.search(r"\w", s)]


def parse_json(text):
    # Returns (dict or None, only_json): only_json is False when there is other text around it.
    stripped = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    match = re.search(r"\{.*\}", stripped, re.DOTALL)
    if not match:
        return None, False
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None, False
    if not isinstance(data, dict):
        return None, False
    return data, stripped == match.group(0)


def norm(value):
    # "Library_Circulation_Log" and "Library Circulation Log" count as the same value.
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def clarity_points(text, truncated):
    return (0 if is_looping(text) else 1) + (0 if truncated else 1)


def score_json(prompt, text, truncated):
    kind, truth = prompt["type"], prompt["truth"]
    data, only_json = parse_json(text)
    s = {"adherence": 0.0, "accuracy": 0.0, "clarity": clarity_points(text, truncated),
         "structure": 1.0 if data is not None else 0.0}
    if data is None:
        return s
    allowed_ok = all(norm(data.get(k, "")) in {norm(a) for a in v} for k, v in ALLOWED[kind].items())
    s["adherence"] = (1.0 if set(data) == KEYS[kind] else 0.0) + (1.0 if allowed_ok else 0.0) \
        + (1.0 if only_json else 0.0)
    if kind == "Classification" and len(re.findall(r"[A-Za-z]+", str(data.get("reason", "")))) < 3:
        s["clarity"] = max(0, s["clarity"] - 1)  # the reason must be a real sentence
    matches = [norm(data.get(k, "")) == norm(v) for k, v in truth.items()]
    s["accuracy"] = 4.0 * sum(matches) / len(matches)
    return s


def copy_ratio(text, source):
    # Share of the answer's 5-word sequences that appear word for word in the input.
    def grams(t):
        w = re.findall(r"\w+", t.lower())
        return {tuple(w[i:i + 5]) for i in range(len(w) - 4)}
    answer = grams(text)
    return len(answer & grams(source)) / len(answer) if answer else 0.0


def facts_found(text, facts):
    return sum(bool(re.search(f, text, re.IGNORECASE)) for f in facts) / len(facts)


def score_summary(prompt, text, truncated):
    n_sent, n_words = len(sentences(text)), len(re.findall(r"\w+", text))
    prose = not re.search(r"^\s*([-*•]|\d+[.)])\s", text, re.MULTILINE) and "{" not in text
    return {"adherence": (1.5 if 1 <= n_sent <= 2 else 0.0) + (1.5 if 0 < n_words <= 60 else 0.0),
            "accuracy": 0.0 if copy_ratio(text, prompt["input"]) > 0.5
            else 4.0 * facts_found(text, prompt["truth"]["facts"]),
            "clarity": clarity_points(text, truncated),
            "structure": 1.0 if prose and text.strip() else 0.0}


def score_email(prompt, text, truncated):
    body = text.strip()
    n_sent, n_words = len(sentences(body)), len(re.findall(r"\w+", body))
    starts_hello = bool(re.match(r"^\W*hello\b", body, re.IGNORECASE))
    signs_off = "datapact" in body[-80:].lower()
    return {"adherence": (1.5 if 3 <= n_sent <= 7 else 0.0) + (1.5 if 0 < n_words <= 130 else 0.0),
            "accuracy": 0.0 if copy_ratio(body, prompt["input"]) > 0.5
            else 4.0 * facts_found(body, prompt["truth"]["facts"]),
            "clarity": clarity_points(body, truncated),
            "structure": 0.5 * starts_hello + 0.5 * signs_off}


def evaluate(prompt, text, truncated=False):
    kind = prompt["type"]
    if kind in KEYS:
        s = score_json(prompt, text, truncated)
    elif kind == "Summarization":
        s = score_summary(prompt, text, truncated)
    else:
        s = score_email(prompt, text, truncated)
    s["score"] = round(max(1.0, sum(s.values())), 1)
    return s


# --- Test the rubric on hand-written answers before running any model ---
P = {p["id"]: p for p in PROMPTS}
if not SMOKE_TEST:
    checks = [
        ("C1", '{"likely_cause": "DATA_ISSUE", "reason": "Prices cannot be negative."}', 10),
        ("C1", '{"likely_cause": "CONTRACT_UPDATED", "reason": "This looks like a new pricing policy."}', 6),
        ("C1", 'Sure! ```json\n{"likely_cause": "DATA_ISSUE", "reason": "The prices are invalid."}\n```', 9),
        ("C1", "The data is wrong. The data is wrong. The data is wrong. The data is wrong. "
               "The data is wrong. The data is wrong.", 1),
        ("E1", '{"dataset": "Library Circulation Log", "column": "checkout_id", '
               '"rule_type": "UNIQUE", "severity": "ERROR"}', 10),
        ("E1", '{"dataset": "Library Circulation Log", "column": "checkout_id", '
               '"rule_type": "DUPLICATE", "severity": "ERROR"}', 8),
        ("E1", '{"dataset": "Library_Circulation_Log", "column": "checkout_id", '
               '"rule_type": "unique", "severity": "ERROR"}', 10),
        ("C1", '{"likely_cause": "DATA_ISSUE", "reason": "CONTRACT_UPDATED", "sample_values": [1]}', 8),
        ("S1", P["S1"]["input"], 3.5),
        ("S1", "Run #412 of the Monthly Enrollment Export failed because 1 rule broke: "
               "47 rows had credit_hours outside 0-24.", 10),
        ("P1", '{"trend": "DECLINING", "next_run": "FAIL"}', 10),
        ("P1", '{"trend": "IMPROVING", "next_run": "FAIL"}', 8),
        ("G1", "Hello, our check of the Weekly Sales Extract found 23 rows where unit_price is zero "
               "or negative. Please correct these prices and resend the file. Thank you for your "
               "help. Best regards, DataPact", 10),
    ]
    for pid, answer, expected in checks:
        got = evaluate(P[pid], answer)["score"]
        assert got == expected, (pid, answer[:40], got, expected)
    print(f"Rubric self-test passed on {len(checks)} hand-written answers.")

## Downloading, loading and running a model

Same approach as A6: each model is downloaded right before its test, models whose files are
over 1 GB are deleted straight after, stalled downloads are retried, and each model is loaded
in 16-bit floats straight onto the GPU. **Greedy decoding** (`do_sample=False`) makes every
answer repeatable.

**Speed measurements** (the model is warmed up with one short generation first, so loading
costs are not counted):

* **Latency**: seconds from sending the prompt to receiving the complete answer.
* **Time to first token (TTFT)**: seconds until the first new token, measured by generating
  exactly one token. This is what a user waits for before anything appears.
* **Tokens/sec**: new tokens divided by latency.
* **Peak GPU memory**: the most GPU memory the model used (CUDA only).

In [ ]:
import gc
import shutil
import time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
from huggingface_hub import scan_cache_dir, snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

WEIGHT_FILES = ["*.json", "*.safetensors", "*.model", "*.txt", "*.py", "*.tiktoken"]
DELETE_ABOVE_GB = 1.0


def download(model_id, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return snapshot_download(model_id, allow_patterns=WEIGHT_FILES)
        except Exception as error:
            print(f"   download attempt {attempt} failed: {type(error).__name__}")
            if attempt == attempts:
                raise
            time.sleep(10)


def tidy_up(model_id):
    for repo in scan_cache_dir().repos:
        if repo.repo_id == model_id and repo.size_on_disk / 1e9 > DELETE_ABOVE_GB:
            scan_cache_dir().delete_revisions(*[r.commit_hash for r in repo.revisions]).execute()
            print(f"   deleted its {repo.size_on_disk / 1e9:.1f} GB of files")


def load(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = {"dtype": torch.float16 if DEVICE != "cpu" else torch.float32}
    if DEVICE == "cuda":
        kwargs["device_map"] = "cuda"
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if DEVICE != "cuda":
        model.to(DEVICE)
    model.eval()
    return tokenizer, model


def unload(model):
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()


def build_messages(prompt):
    return [{"role": "user", "content": INSTRUCTIONS[prompt["type"]] + "\n\n" + prompt["input"]}]


def timed_generate(tokenizer, model, inputs, max_new_tokens):
    sync()
    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id)
    sync()
    return output, time.perf_counter() - start


def run_model(model_id, category, params):
    tokenizer, model = load(model_id)
    # Warm-up so the first prompt does not pay one-off start-up costs.
    warm = tokenizer.apply_chat_template([{"role": "user", "content": "Hello"}],
                                         add_generation_prompt=True, return_tensors="pt",
                                         return_dict=True).to(model.device)
    timed_generate(tokenizer, model, warm, 4)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    rows = []
    for prompt in PROMPTS:
        inputs = tokenizer.apply_chat_template(build_messages(prompt), add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model.device)
        n_prompt = inputs["input_ids"].shape[1]
        limit = MAX_NEW_TOKENS[prompt["type"]]
        _, ttft = timed_generate(tokenizer, model, inputs, 1)
        output, seconds = timed_generate(tokenizer, model, inputs, limit)
        new_tokens = output[0][n_prompt:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        truncated = len(new_tokens) >= limit
        rows.append({
            "Model": model_id.split("/")[-1], "model_id": model_id, "Model Category": category,
            "Parameters (B)": params, "Prompt Type": prompt["type"], "Prompt ID": prompt["id"],
            "Prompt": prompt["input"], "prompt_tokens": n_prompt, "new_tokens": len(new_tokens),
            "Latency (s)": round(seconds, 3), "TTFT (s)": round(ttft, 3),
            "Tokens/sec": round(len(new_tokens) / seconds, 1) if seconds else None,
            "truncated": truncated, "output": text, **evaluate(prompt, text, truncated),
        })
    peak = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None
    for row in rows:
        row["Peak GPU memory (GB)"] = round(peak, 2) if peak else None
    unload(model)
    return rows

## Step 1.5: Run the benchmark (15 models x 25 prompts)

Each model is loaded once and answers all 25 prompts. The results file is rewritten after every
model, so a crash only loses the model that was running; *Run all* again resumes.

In [ ]:
os.makedirs("results", exist_ok=True)
BENCH_CSV = "results/benchmark_results.csv"

rows = pd.read_csv(BENCH_CSV).to_dict("records") if os.path.exists(BENCH_CSV) else []
done = {r["model_id"] for r in rows}
if done:
    print("Resuming: already finished", len(done), "models")

for category, model_id, params in MODELS:
    if model_id in done:
        continue
    print(f"[{category}] {model_id} ...", flush=True)
    try:
        download(model_id)
        rows += run_model(model_id, category, params)
    except Exception as error:  # one broken model should not stop the rest
        print("   failed:", type(error).__name__, error)
    tidy_up(model_id)
    pd.DataFrame(rows).to_csv(BENCH_CSV, index=False)

bench = pd.DataFrame(rows)
bench = bench[bench["model_id"].isin({m[1] for m in MODELS})].reset_index(drop=True)
bench.to_csv(BENCH_CSV, index=False)
print("Saved", BENCH_CSV, "with", len(bench), "rows from", bench["model_id"].nunique(), "models")

### The Step 1.5 results table

One row per model and prompt (375 rows). The full table, including every model's raw output and
the four sub-scores, is saved in `results/benchmark_results.csv`.

In [ ]:
TABLE_COLUMNS = ["Model", "Model Category", "Tokens/sec", "Latency (s)", "Prompt Type",
                 "Prompt ID", "Prompt", "score"]
step_1_5 = bench[TABLE_COLUMNS].rename(columns={"score": "Self-Evaluation Score (1-10)"})
step_1_5["Prompt"] = step_1_5["Prompt"].str.replace("\n", " | ")
step_1_5.to_csv("results/step1_5_results_table.csv", index=False)
with pd.option_context("display.max_rows", 400, "display.max_colwidth", 70):
    display(step_1_5)

### Summaries

Averages per model, per category and per prompt type, used for the analysis in Step 1.6 and
for the cost model in Part 2.

In [ ]:
def summarise(frame, by):
    out = frame.groupby(by, sort=False).agg(
        avg_score=("score", "mean"), adherence=("adherence", "mean"), accuracy=("accuracy", "mean"),
        clarity=("clarity", "mean"), structure=("structure", "mean"),
        tokens_per_sec=("Tokens/sec", "mean"), latency_s=("Latency (s)", "mean"),
        ttft_s=("TTFT (s)", "mean"), avg_new_tokens=("new_tokens", "mean"),
        avg_prompt_tokens=("prompt_tokens", "mean"), peak_mem_gb=("Peak GPU memory (GB)", "max"))
    return out.round(2)


by_model = summarise(bench, ["Model Category", "Model"]).reset_index()
by_model["Parameters (B)"] = by_model["Model"].map(dict(zip(bench["Model"], bench["Parameters (B)"])))
by_model.sort_values("avg_score", ascending=False)

In [ ]:
by_category = summarise(bench, ["Model Category"]).reindex(
    [c for c in CATEGORY_ORDER if c in set(bench["Model Category"])])
by_category

In [ ]:
# Average score for each prompt type in each category: which tasks need a bigger model?
type_by_category = bench.pivot_table(index="Model Category", columns="Prompt Type", values="score",
                                     aggfunc="mean").reindex(
    [c for c in CATEGORY_ORDER if c in set(bench["Model Category"])])[PROMPT_TYPES].round(1)
type_by_category

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
colours = dict(zip(CATEGORY_ORDER, ["#9aa5b1", "#2f5bd7", "#1d6b36", "#855c11", "#a8323e"]))
for _, row in by_model.iterrows():
    axes[0].scatter(row["tokens_per_sec"], row["avg_score"], s=60 + 60 * row["Parameters (B)"],
                    color=colours[row["Model Category"]], alpha=0.85)
    axes[0].annotate(row["Model"].replace("-Instruct", "").replace("-chat", ""),
                     (row["tokens_per_sec"], row["avg_score"]), fontsize=7,
                     xytext=(4, 3), textcoords="offset points")
for category, colour in colours.items():
    axes[0].scatter([], [], color=colour, label=category)
axes[0].legend(title="Category")
axes[0].set_xlabel("Tokens per second (faster ->)")
axes[0].set_ylabel("Average self-evaluation score (1-10)")
axes[0].set_title("Speed vs quality per model (dot size = parameters)")
axes[0].grid(alpha=0.3)

type_by_category.T.plot.bar(ax=axes[1], color=[colours[c] for c in type_by_category.index])
axes[1].set_ylabel("Average score (1-10)")
axes[1].set_title("Average score by prompt type and category")
axes[1].set_ylim(0, 10)
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(title="Category", fontsize=8)
plt.tight_layout()
plt.savefig("results/speed_vs_quality.png", dpi=120)
plt.show()

### Handoff for Part 2 (Connor): per-model speed and size

`results/model_speed_summary.csv` holds what the cost model needs for every model: tokens per
second and latency on a single T4 GPU, the average prompt and answer lengths on DataPact's
prompts, and peak GPU memory.

In [ ]:
speed_summary = by_model[["Model Category", "Model", "Parameters (B)", "tokens_per_sec", "latency_s",
                          "ttft_s", "avg_prompt_tokens", "avg_new_tokens", "peak_mem_gb", "avg_score"]]
speed_summary.to_csv("results/model_speed_summary.csv", index=False)
speed_summary

In [ ]:
# Sample raw outputs: the best and worst scored answer for each prompt type.
for kind in PROMPT_TYPES:
    subset = bench[bench["Prompt Type"] == kind].sort_values("score")
    for label, row in (("LOWEST", subset.iloc[0]), ("HIGHEST", subset.iloc[-1])):
        print(f"=== {kind} | {label} | {row['Model']} on {row['Prompt ID']} | score {row['score']} ===")
        print(str(row["output"])[:500], "\n")

## Step 1.6: Analysis

*Step 1.6 is Ashok's part. The tables and chart above have the numbers it needs.*

- **Which model category produced the best outputs?** _to fill in_
- **Did larger models always perform better?** _to fill in_
- **Which model gave the best balance between speed and output quality?** _to fill in_

## Download the results (Colab only)

Zips `results/` and downloads it. Then save this notebook with *File > Download > Download .ipynb*
so its outputs are kept.

In [ ]:
if IN_COLAB:
    from google.colab import files
    shutil.make_archive("results", "zip", "results")
    files.download("results.zip")
else:
    print("Not in Colab - results are already in ./results")